# Phase 24 Clustering and Dimensionality Reduction Demo

This notebook visualizes deterministic artifacts generated by `backend/scripts/run_phase24_demo_pipeline.py`.


## 1) Global map (100 users)


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("data")


def load_csv(name: str) -> pd.DataFrame:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}")
    return pd.read_csv(path)


def load_json(name: str) -> dict:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
global_before = load_csv("phase24_global_before.csv")
global_after = load_csv("phase24_global_after.csv")

print(f"Global before rows: {len(global_before)}")
print(f"Global after rows: {len(global_after)}")

In [ ]:
# Defensive loading so this cell can run even if prior cells were skipped.
if "global_before" not in globals() or "global_after" not in globals():
    global_before = load_csv("phase24_global_before.csv")
    global_after = load_csv("phase24_global_after.csv")

required_cols = {"x", "y", "nickname"}
missing_before = required_cols - set(global_before.columns)
missing_after = required_cols - set(global_after.columns)

if missing_before:
    raise ValueError(
        f"phase24_global_before.csv is missing columns: {sorted(missing_before)}. "
        "Regenerate artifacts with: cd backend && ./venv/bin/python scripts/run_phase24_demo_pipeline.py --use-fixture-data"
    )
if missing_after:
    raise ValueError(
        f"phase24_global_after.csv is missing columns: {sorted(missing_after)}. "
        "Regenerate artifacts with: cd backend && ./venv/bin/python scripts/run_phase24_demo_pipeline.py --use-fixture-data"
    )
if len(global_before) == 0 or len(global_after) == 0:
    raise ValueError(
        "Global demo CSVs are empty. Regenerate artifacts with: "
        "cd backend && ./venv/bin/python scripts/run_phase24_demo_pipeline.py --use-fixture-data"
    )

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(global_before["x"], global_before["y"], s=30, alpha=0.65, label="Before")
ax.scatter(global_after["x"], global_after["y"], s=18, alpha=0.45, label="After")
ax.set_title("Global map for all 100 demo users")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.axhline(0.0, color="gray", lw=0.5)
ax.axvline(0.0, color="gray", lw=0.5)
ax.legend()
ax.set_aspect("equal")
ax.set_box_aspect(1)
plt.show()


## 2) Eleanor Colvin ego subset (20 friends)


In [ ]:
ego_before = load_csv("phase24_eleanor_ego_before.csv")
ego_after = load_csv("phase24_eleanor_ego_after.csv")

print(f"Eleanor ego before rows: {len(ego_before)}")
print(f"Eleanor ego after rows: {len(ego_after)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tiers = ego_before["tier"].astype(int)
scatter = ax.scatter(
    ego_before["x"], ego_before["y"], c=tiers, cmap="viridis", s=70, alpha=0.8
)

for _, row in ego_before.iterrows():
    if row["nickname"] in {"Eleanor Colvin", "Winston Churchill"}:
        ax.annotate(
            row["nickname"],
            (row["x"], row["y"]),
            xytext=(6, 6),
            textcoords="offset points",
        )

ax.set_title("Eleanor Colvin ego subset before amplification")
ax.set_xlabel("x (Eleanor-centered)")
ax.set_ylabel("y (Eleanor-centered)")
ax.axhline(0.0, color="gray", lw=0.5)
ax.axvline(0.0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.set_box_aspect(1)
fig.colorbar(scatter, ax=ax, label="tier")
plt.show()

## 3) Eleanor-centered coordinate shift


In [ ]:
shift = load_csv("phase24_eleanor_shift.csv")

top_shift = shift.assign(
    distance=((shift["delta_x"] ** 2 + shift["delta_y"] ** 2) ** 0.5)
)
top_shift = top_shift.sort_values("distance", ascending=False).head(10)
top_shift[["nickname", "delta_x", "delta_y", "distance"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.quiver(
    shift["before_x"],
    shift["before_y"],
    shift["delta_x"],
    shift["delta_y"],
    angles="xy",
    scale_units="xy",
    scale=1,
    alpha=0.55,
)
ax.set_title("Eleanor-centered coordinate shift vectors (before -> after)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.axhline(0.0, color="gray", lw=0.5)
ax.axvline(0.0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.set_box_aspect(1)
plt.show()

## 4) Side-by-side before/after Eleanor map with boosted Eleanor<->Winston likes/comments


In [ ]:
side_by_side = load_json("phase24_eleanor_side_by_side.json")
before_df = pd.DataFrame(side_by_side["before"])
after_df = pd.DataFrame(side_by_side["after"])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, frame, label in [
    (axes[0], before_df, "Before amplification"),
    (axes[1], after_df, "After amplification"),
]:
    ax.scatter(frame["x"], frame["y"], c=frame["tier"], cmap="plasma", s=80, alpha=0.85)
    for _, row in frame.iterrows():
        if row["nickname"] in {"Eleanor Colvin", "Winston Churchill"}:
            ax.annotate(
                row["nickname"],
                (row["x"], row["y"]),
                xytext=(6, 6),
                textcoords="offset points",
            )
    ax.set_title(label)
    ax.axhline(0.0, color="gray", lw=0.5)
    ax.axvline(0.0, color="gray", lw=0.5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1)


# keep both panels on the same coordinate window for fair visual comparison
combined_x = pd.concat([before_df["x"], after_df["x"]])
combined_y = pd.concat([before_df["y"], after_df["y"]])
pad_x = (combined_x.max() - combined_x.min()) * 0.08 if len(combined_x) else 0.1
pad_y = (combined_y.max() - combined_y.min()) * 0.08 if len(combined_y) else 0.1
x_min, x_max = combined_x.min() - pad_x, combined_x.max() + pad_x
y_min, y_max = combined_y.min() - pad_y, combined_y.max() + pad_y
for ax in axes:
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

fig.suptitle("Eleanor map before/after interaction amplification")
axes[0].set_xlabel("x")
axes[1].set_xlabel("x")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

side_by_side["comparison"]

## 5) Euclidean distance proof (Eleanor vs Winston)

This section computes the direct Euclidean distance between **Eleanor Colvin** and **Winston Churchill** in the Eleanor-centered ego map, before and after amplification. A shorter distance after amplification confirms the stronger interaction pulled Winston closer in embedding space.


In [ ]:
pair = {"a": "Eleanor Colvin", "b": "Winston Churchill"}


def euclidean_between(df, a, b):
    row_a = df.loc[df["nickname"] == a, ["x", "y"]].iloc[0]
    row_b = df.loc[df["nickname"] == b, ["x", "y"]].iloc[0]
    dx = float(row_a["x"] - row_b["x"])
    dy = float(row_a["y"] - row_b["y"])
    return (dx**2 + dy**2) ** 0.5


d_before = euclidean_between(ego_before, pair["a"], pair["b"])
d_after = euclidean_between(ego_after, pair["a"], pair["b"])

comparison = pd.DataFrame(
    [
        {"scenario": "Before amplification", "euclidean_distance": d_before},
        {"scenario": "After amplification", "euclidean_distance": d_after},
    ]
)
comparison["delta_vs_before"] = comparison["euclidean_distance"] - d_before

print(f"Distance({pair['a']} ↔ {pair['b']}) BEFORE: {d_before:.6f}")
print(f"Distance({pair['a']} ↔ {pair['b']}) AFTER : {d_after:.6f}")
print(f"Change (after - before): {d_after - d_before:+.6f}")

ax = comparison.plot(
    kind="bar",
    x="scenario",
    y="euclidean_distance",
    legend=False,
    figsize=(6, 6),
    color=["#4c78a8", "#f58518"],
)
ax.set_ylabel("Euclidean distance")
ax.set_title("Eleanor ↔ Winston distance before/after amplification")
ax.set_box_aspect(1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.6f")
plt.tight_layout()
plt.show()

comparison


## 6) Distance trend across amplification levels

This section shows how **Eleanor ↔ Winston distance** changes as we increase interaction amplification.
For stakeholders: a lower value means the pair is closer in embedding space after stronger interaction signals.


In [ ]:
distance_curve = load_csv("phase24_eleanor_winston_distance_curve.csv").sort_values("amplification_likes")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    distance_curve["amplification_likes"],
    distance_curve["euclidean_distance"],
    marker="o",
    linewidth=2,
)
ax.set_title("Eleanor ↔ Winston distance vs interaction amplification")
ax.set_xlabel("Amplification likes delta")
ax.set_ylabel("Euclidean distance")
ax.grid(True, alpha=0.25)
plt.show()

distance_curve[[
    "amplification_likes",
    "amplification_comments",
    "euclidean_distance",
    "distance_delta_from_baseline",
]].head(8)


## 7) Rank and force diagnostics across amplification levels

Phase 25 adds explainability fields that show *why* movement does or does not change:
- `nearest_neighbor_rank` and `rank_delta_from_baseline`
- `effective_pull` and `effective_pull_delta_from_baseline`
- `movement_explanation` narrative strings

For non-technical readers: rank tells us relative closeness ordering, while effective pull reflects the strength of attraction after sensitivity scaling and caps.


In [ ]:
distance_curve = load_csv("phase24_eleanor_winston_distance_curve.csv").sort_values("amplification_likes")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    distance_curve["amplification_likes"],
    distance_curve["nearest_neighbor_rank"],
    marker="o",
    color="tab:orange",
)
axes[0].set_title("Nearest-neighbor rank across amplification")
axes[0].set_xlabel("Amplification likes delta")
axes[0].set_ylabel("Nearest-neighbor rank")
axes[0].grid(True, alpha=0.25)

axes[1].plot(
    distance_curve["amplification_likes"],
    distance_curve["effective_pull"],
    marker="o",
    color="tab:green",
)
axes[1].set_title("Effective pull across amplification")
axes[1].set_xlabel("Amplification likes delta")
axes[1].set_ylabel("Effective pull")
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

distance_curve[[
    "amplification_likes",
    "nearest_neighbor_rank",
    "rank_delta_from_baseline",
    "effective_pull",
    "effective_pull_delta_from_baseline",
    "movement_explanation",
]]


## 8) Safeguards and interpretation notes for non-technical review

- **Guardrail remains on:** movement clipping is still enabled, even in the demo-strong preset.
- **How to read stable rows:** if rank and distance remain flat while effective pull rises, other neighborhood constraints can dominate.
- **Operational guidance:** always compare production-safe default first, then demo-strong sensitivity, and explain changes using distance + rank + force together.
